# Step 9: Garment-detail term extraction

This reuses the exact zero-shot mechanic from notebook 05 (comparing the vibe vector against text embeddings via cosine similarity), but with a very different vocabulary. Step 7's words ("minimalist", "cottagecore") describe a broad *aesthetic* -- useful for tagging, but useless as a Depop search term ("y2k" surfaces almost nothing specific). This notebook's vocabulary is concrete, physical garment attributes -- "cropped", "sheer", "off the shoulder" -- exactly the kind of words that make a Depop search actually return relevant listings.

## Load the vibe vector and FashionCLIP

Self-contained, same as previous notebooks.

In [1]:
from pathlib import Path
import numpy as np
import torch
from transformers import CLIPModel, CLIPProcessor

vibe_vector = np.load(Path("../data/embeddings/board_vibe_vector.npy"))

MODEL_NAME = "patrickjohncyh/fashion-clip"
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
model.eval()
processor = CLIPProcessor.from_pretrained(MODEL_NAME)

print("Model loaded. Vibe vector shape:", vibe_vector.shape)

Using device: mps


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model loaded. Vibe vector shape: (512,)


## The garment-detail vocabulary

Organized into four groups, per the project scope -- we keep the group labels around for display, but ranking happens across *all* terms together, regardless of group, since a Depop query doesn't care whether "sheer" is a fabric term or "cropped" is a fit term.

In [2]:
GARMENT_VOCAB = {
    "Silhouette/fit": ["oversized", "cropped", "fitted", "high-waisted", "wide-leg", "bodycon", "baggy"],
    "Neckline/sleeve": ["off the shoulder", "halter", "puff sleeve", "spaghetti strap", "turtleneck", "square neck"],
    "Fabric/texture": ["sheer", "satin", "corduroy", "mesh", "knit", "denim", "leather", "velvet"],
    "Pattern/finish": ["pastel", "plaid", "floral", "metallic", "distressed", "ribbed"],
}

# Flatten into two parallel lists: every term, and which group it came from
terms = []
term_groups = []
for group, words in GARMENT_VOCAB.items():
    for word in words:
        terms.append(word)
        term_groups.append(group)

print(f"Total terms: {len(terms)}")

Total terms: 27


## Embed the vocabulary

Same prompt-engineering idea as notebook 05 -- wrap each bare word in a template sentence rather than embedding it alone. We need a template general enough to read naturally across all four groups (a fit term like "cropped", a fabric term like "sheer", a multi-word neckline term like "off the shoulder"): `"a photo of clothing that is {term}"` works reasonably for all of them.

Same output-object detail as before: the real embedding is at `.pooler_output`.

In [3]:
PROMPT_TEMPLATE = "a photo of clothing that is {}"
prompts = [PROMPT_TEMPLATE.format(term) for term in terms]

inputs = processor(text=prompts, return_tensors="pt", padding=True).to(device)

with torch.no_grad():
    outputs = model.get_text_features(**inputs)

term_embeddings = outputs.pooler_output
term_embeddings = term_embeddings / term_embeddings.norm(dim=1, keepdim=True)

print("Term embeddings shape:", term_embeddings.shape)

Term embeddings shape: torch.Size([27, 512])


## Rank every term against the vibe vector

Same math as notebook 05: both sides are unit-length, so the dot product is the cosine similarity. Per the project scope, we take the top 3-5 overall as the actual Depop search keywords.

In [4]:
vibe_tensor = torch.from_numpy(vibe_vector).to(device)
similarities = term_embeddings @ vibe_tensor

TOP_K = 5
sorted_indices = torch.argsort(similarities, descending=True)

print("All terms, ranked:\n")
for idx in sorted_indices:
    marker = "*" if idx in sorted_indices[:TOP_K] else " "
    print(f" {marker} {terms[idx]:<20} ({term_groups[idx]:<16}) {similarities[idx].item():.4f}")

top_terms = [terms[idx] for idx in sorted_indices[:TOP_K]]
print(f"\nTop {TOP_K} garment-detail terms: {top_terms}")

All terms, ranked:

 * cropped              (Silhouette/fit  ) 0.3497
 * square neck          (Neckline/sleeve ) 0.3304
 * off the shoulder     (Neckline/sleeve ) 0.3296
 * puff sleeve          (Neckline/sleeve ) 0.3251
 * high-waisted         (Silhouette/fit  ) 0.3224
   sheer                (Fabric/texture  ) 0.3075
   halter               (Neckline/sleeve ) 0.3054
   ribbed               (Pattern/finish  ) 0.3053
   distressed           (Pattern/finish  ) 0.3024
   baggy                (Silhouette/fit  ) 0.3013
   oversized            (Silhouette/fit  ) 0.3003
   mesh                 (Fabric/texture  ) 0.2960
   spaghetti strap      (Neckline/sleeve ) 0.2952
   fitted               (Silhouette/fit  ) 0.2940
   metallic             (Pattern/finish  ) 0.2883
   wide-leg             (Silhouette/fit  ) 0.2867
   denim                (Fabric/texture  ) 0.2855
   velvet               (Fabric/texture  ) 0.2816
   satin                (Fabric/texture  ) 0.2812
   knit                 (Fabri

## Per-category plausibility gate

The ranked list above is board-level — it says *this board* leans "cropped / square neck / off the shoulder". But notebook 08 builds one Depop query per recommended item, and every item has a category: you don't want "square neck" attached to a pair of shorts.

So we build a **plausibility gate** — for every clothing category in the pool, which of the 27 garment terms actually apply to it. It's derived from the item pool itself, not hand-written.

In [5]:
import pandas as pd

item_pool_embeddings = np.load(Path("../data/embeddings/item_embeddings.npy"))
item_pool_ids = np.load(Path("../data/embeddings/item_ids.npy"))

dataset_root = Path.home() / ".cache/kagglehub/datasets/paramaggarwal/fashion-product-images-small/versions/1"
styles_df = pd.read_csv(dataset_root / "styles.csv", on_bad_lines="skip").set_index("id")

item_categories = styles_df.reindex(item_pool_ids)["articleType"].values
print(f"Loaded {len(item_pool_embeddings)} item photos across {pd.Series(item_categories).nunique()} categories")

Loaded 44419 item photos across 142 categories


In [6]:
MIN_PHOTOS_PER_CATEGORY = 20  # fewer than this and the averaged centroid is too noisy to trust

category_counts = pd.Series(item_categories).value_counts()

category_centroids = {}
for category, count in category_counts.items():
    if count < MIN_PHOTOS_PER_CATEGORY:
        continue
    mask = item_categories == category
    centroid = item_pool_embeddings[mask].mean(axis=0)
    centroid = centroid / np.linalg.norm(centroid)
    category_centroids[category] = centroid

skipped = int((category_counts < MIN_PHOTOS_PER_CATEGORY).sum())
print(f"Built centroids for {len(category_centroids)} categories "
      f"(skipped {skipped} with fewer than {MIN_PHOTOS_PER_CATEGORY} real photos)")

Built centroids for 90 categories (skipped 52 with fewer than 20 real photos)


### Build the gate

Two signals, both grounded in the dataset:

1. **subCategory hard rule** — a category whose dominant `styles.csv` subCategory isn't Topwear / Innerwear / Dress physically has no neckline or sleeve, so every Neckline/sleeve term is dropped from it. (This is what stops "square neck shorts".)
2. **Centroid median** — build each category's centroid (the mean embedding of its real photos), score every term against it, and keep only terms at or above that category's own median term score.

In [7]:
category_names = list(category_centroids.keys())
centroid_matrix = np.stack([category_centroids[c] for c in category_names])
category_term_scores = term_embeddings.detach().cpu().numpy() @ centroid_matrix.T

NECKLINE_CAPABLE_SUBCATEGORIES = {"Topwear", "Innerwear", "Dress"}
NECKLINE_SLEEVE_TERMS = {terms[i] for i in range(len(terms)) if term_groups[i] == "Neckline/sleeve"}
# Dominant subCategory for each articleType, straight from styles.csv's own labels.
category_subcategory = styles_df.groupby("articleType")["subCategory"].agg(lambda s: s.value_counts().idxmax())

category_gate = {}
for col, category in enumerate(category_names):
    scores = category_term_scores[:, col]
    threshold = float(np.median(scores))
    passed_terms = [terms[i] for i in range(len(terms)) if scores[i] >= threshold]
    if category_subcategory.get(category) not in NECKLINE_CAPABLE_SUBCATEGORIES:
        passed_terms = [t for t in passed_terms if t not in NECKLINE_SLEEVE_TERMS]
    category_gate[category] = {"threshold": threshold, "passed_terms": passed_terms}

print(f"Gate built for {len(category_gate)} categories")

Gate built for 90 categories


### Sanity-check the gate

Spot-check a few categories: `Camisoles` should keep strappy / neckline terms, `Shorts` should keep fit terms but drop every neckline term, and `Culottes` may have too few real photos for a centroid at all.

In [8]:
for category in ["Camisoles", "Shorts", "Dresses", "Culottes"]:
    gate = category_gate.get(category)
    if gate is None:
        print(f"{category}: no centroid (too few real photos in the dataset)\n")
        continue
    dropped = [t for t in terms if t not in gate["passed_terms"]]
    print(f"{category}  (threshold={gate['threshold']:.4f})")
    print(f"  passed:  {gate['passed_terms']}")
    print(f"  dropped: {dropped}\n")

Camisoles  (threshold=0.2581)
  passed:  ['cropped', 'fitted', 'high-waisted', 'bodycon', 'off the shoulder', 'halter', 'puff sleeve', 'spaghetti strap', 'square neck', 'sheer', 'satin', 'mesh', 'velvet', 'ribbed']
  dropped: ['oversized', 'wide-leg', 'baggy', 'turtleneck', 'corduroy', 'knit', 'denim', 'leather', 'pastel', 'plaid', 'floral', 'metallic', 'distressed']

Shorts  (threshold=0.2467)
  passed:  ['oversized', 'cropped', 'fitted', 'high-waisted', 'wide-leg', 'baggy', 'sheer', 'satin', 'corduroy', 'mesh', 'distressed', 'ribbed']
  dropped: ['bodycon', 'off the shoulder', 'halter', 'puff sleeve', 'spaghetti strap', 'turtleneck', 'square neck', 'knit', 'denim', 'leather', 'velvet', 'pastel', 'plaid', 'floral', 'metallic']

Dresses  (threshold=0.2718)
  passed:  ['oversized', 'cropped', 'fitted', 'bodycon', 'off the shoulder', 'halter', 'puff sleeve', 'spaghetti strap', 'square neck', 'sheer', 'satin', 'mesh', 'floral', 'ribbed']
  dropped: ['high-waisted', 'wide-leg', 'baggy', 't

## Save for notebook 08

Notebook 08 combines these with each recommended item's category to build the Depop search URLs. We persist three things so that step never needs to reload FashionCLIP:

- `top_terms` — the board's top garment-detail terms
- `all_terms_ranked` — every term with its group and board-relevance score
- `category_gate` — the per-category plausibility filter

In [9]:
import json

all_terms_ranked = [
    {"term": terms[idx], "group": term_groups[idx], "score": similarities[idx].item()}
    for idx in sorted_indices
]

output_path = Path("../data/embeddings/garment_terms.json")
with open(output_path, "w") as f:
    json.dump(
        {"top_terms": top_terms, "all_terms_ranked": all_terms_ranked, "category_gate": category_gate},
        f,
        indent=2,
    )

print(f"Saved {len(all_terms_ranked)} ranked terms + a plausibility gate for "
      f"{len(category_gate)} categories to {output_path}")
print("Shorts    ->", category_gate["Shorts"]["passed_terms"])
print("Camisoles ->", category_gate["Camisoles"]["passed_terms"])

Saved 27 ranked terms + a plausibility gate for 90 categories to ../data/embeddings/garment_terms.json
Shorts    -> ['oversized', 'cropped', 'fitted', 'high-waisted', 'wide-leg', 'baggy', 'sheer', 'satin', 'corduroy', 'mesh', 'distressed', 'ribbed']
Camisoles -> ['cropped', 'fitted', 'high-waisted', 'bodycon', 'off the shoulder', 'halter', 'puff sleeve', 'spaghetti strap', 'square neck', 'sheer', 'satin', 'mesh', 'velvet', 'ribbed']
